In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from matplotlib import pyplot as plt
%matplotlib inline
import seaborn as sns

# Scaling libraries
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

In [2]:
df_unknown = pd.read_csv('TTGGG_K.csv')
# df_unknown_new = df_unknown.loc[:,['H8','H1p']]
#df_unknown.dropna(axis=0, inplace=True)
df_unknown_parallel = df_unknown.loc[0:11,['S.No.', 'PDB_ID', 'PLANE', 'POSITION','H1p','H8','H2p','H2pp','H1']]
df_unknown_parallel

,S.No.,PDB_ID,PLANE,POSITION,H1p,H8,H2p,H2pp,H1
0,10025,TTGGG_K+,4,X1,5.93,7.51,2.38,2.65,10.72
1,10026,TTGGG_K+,4,X2,6.21,7.74,2.54,2.77,10.89
2,10027,TTGGG_K+,4,X3,6.22,7.87,2.51,2.75,11.12
3,10028,TTGGG_K+,4,X4,6.21,7.89,2.51,2.76,11.21
4,10029,TTGGG_K+,4,X5,5.98,7.68,2.65,2.75,11.30
5,10030,TTGGG_K+,4,X6,6.05,8.01,2.46,2.81,11.32
6,10031,TTGGG_K+,4,X7,6.19,7.73,2.70,2.85,11.43
7,10032,TTGGG_K+,4,X8,6.14,7.69,2.61,2.79,11.46
8,10033,TTGGG_K+,4,X9,6.18,7.70,2.66,2.81,11.55
9,10034,TTGGG_K+,4,X10,6.14,8.03,2.50,2.93,11.58


In [3]:
### Importing the initial dataframe from excel file
df_parallel = pd.read_excel('CS_MKM_20200609.xlsx', sheet_name='parallel 12to0')

### Extracting the required columns to a new dataframe
df_parallel_new = df_parallel.loc[:,['S.No.', 'PDB_ID', 'PLANE', 'POSITION','H1p','H8','H2p','H2pp','H1']]
print("Shape before removing the NAN values:", df_parallel_new.shape)

### Removing NAN rows : here it is the gap between each sample entry (after 12 nucleotides). There are no 
### NAN values in the data itself.
df_parallel_new.dropna(axis=0, inplace=True)
print("Shape after removing the NAN values:", df_parallel_new.shape)

### Removing 2lby from the list because it is coordinated with ligand and not a free quadruplex
df_parallel_new.drop(df_parallel_new.loc[df_parallel_new['PDB_ID'] == '2lby'].index, inplace=True)
print("After removing data which doesn't involve free G quad:",df_parallel_new.shape)

Shape before removing the NAN values: (323, 9)
Shape after removing the NAN values: (203, 9)
After removing data which doesn't involve free G quad: (192, 9)


## The Final Frontier - TTGGG

In [14]:
%%time
import os
filename = os.path.expanduser('~') + '/TTGGG_linear_TEST_13_5H_C_1.out' #15
try:
    os.remove(filename)
except OSError:
    pass
f1 = open(filename, 'a')

### Importing the initial dataframe from excel file
df_parallel = pd.read_excel('CS_MKM_20200609.xlsx', sheet_name='parallel 12to0')

### Extracting the required columns to a new dataframe
df_parallel_new = df_parallel.loc[:,['S.No.', 'PDB_ID', 'PLANE', 'POSITION','H1p','H8','H2p','H2pp','H1']]
print("Shape before removing the NAN values:", df_parallel_new.shape)

### Removing NAN rows : here it is the gap between each sample entry (after 12 nucleotides). There are no 
### NAN values in the data itself.
df_parallel_new.dropna(axis=0, inplace=True)
print("Shape after removing the NAN values:", df_parallel_new.shape)

### Removing 2lby from the list because it is coordinated with ligand and not a free quadruplex
df_parallel_new.drop(df_parallel_new.loc[df_parallel_new['PDB_ID'] == '2lby'].index, inplace=True)
print("After removing data which doesn't involve free G quad:",df_parallel_new.shape)

### Sp, split df_parallel_new dataframe into 19 smaller ones
df_split = np.array_split(df_parallel_new, 16) #18

y_pred_list = [] # Holding all prediction lists
#PDB_ID_chosen_list = [] # Holding all unique PDB_ids
accuracy_list = [] # Holding all accuracy lists

for i in range(1000):
    ### Random selection of 15 out of 18 dataframes and concating them into a single dataframe df_selected
    import random
    sampled_list = random.sample(df_split, 13) #15
    df_selected = pd.concat(sampled_list)
    # print(df_selected.shape)

    ### Randomizing the input 15 dataframes/samples
    df_selected = df_selected.sample(frac=1)

    ### Now, we have a total of 15 samples in df_selected dataframe
    ### Adding 1XAV to the df_selected towards the end

    df_final = pd.concat([df_selected,df_unknown_parallel])
    # print(df_final['PDB_ID'].unique())
    ### Now we have a total of 16 samples where prediction need to be performed for 16 samples

    X = df_final.iloc[:,[4,5,6,7,8]]
    Y = df_final['PLANE']

    ### train_test_split on df_final with 16 samples

    X_train, X_test, y_train, y_test = train_test_split(X,
                                                    Y,
                                                    test_size=12/168, 
                                                    shuffle=False) 
    # Shuffle takes priority over random_state. If shuffle is set to FALSE, all the n first observations in your dataframe will 
    # go in the train dataset, and all others in the test dataset.
    # By default, shuffle is set to TRUE . So a random state is used before splitting.
    # Random state = None (default). It means np.random.

    # print(X_test.shape)
    ### Scaling using min_max scale
    sc = StandardScaler()
    X_train = sc.fit_transform(X_train)
    X_test = sc.transform (X_test) 

    #Import svm model
    from sklearn import svm

    #Create a svm Classifier
    clf = svm.SVC(kernel='linear', gamma='scale', C=1) 

    #Train the model using the training sets
    clf.fit(X_train, y_train)

    #Predict the response for test dataset
    y_pred = clf.predict(X_test)

    #Import scikit-learn metrics module for accuracy calculation
    from sklearn import metrics

    # Model Accuracy: how often is the classifier correct?
    # print("Accuracy",i,"run:",metrics.accuracy_score(y_test, y_pred))

    # Appending the results of each iteration to a list 
    y_pred_list.append(y_pred)

    #PDB_ID_chosen_list.append(df_final['PDB_ID'].unique())
    #accuracy_list.append(metrics.accuracy_score(y_test, y_pred))

y_test = y_test.reset_index()
df_pred = pd.DataFrame(y_pred_list)
df_pred = df_pred.T
df_pred['Predicted_1'] = (df_pred == 1).astype(int).sum(axis=1)
df_pred['Predicted_2'] = (df_pred == 2).astype(int).sum(axis=1)
df_pred['Predicted_3'] = (df_pred == 3).astype(int).sum(axis=1)

y_test['PDB_ID'] = df_unknown_parallel['PDB_ID'].values
y_test['POSITION'] = df_unknown_parallel['POSITION'].values
y_test['H1p'] = df_unknown_parallel['H1p'].values
y_test['H8'] = df_unknown_parallel['H8'].values
y_test['H2p'] = df_unknown_parallel['H2p'].values
y_test['H2pp'] = df_unknown_parallel['H2pp'].values
y_test['H1'] = df_unknown_parallel['H1'].values
# Changing the order of columns in a dataframe
y_test.drop(['index'],axis=1, inplace =True)
y_test['S.No.'] = df_unknown_parallel['S.No.']
y_test = y_test[['S.No.','PDB_ID','PLANE','POSITION','H1p','H8','H2p','H2pp','H1']]
y_test['Count(1)'] = df_pred['Predicted_1']
y_test['Count(2)'] = df_pred['Predicted_2']
y_test['Count(3)'] = df_pred['Predicted_3']

y_test.to_csv('~/TTGGG_linear_TEST_13_5H_C_1.out',sep=',', index=False, header=True) #15

Shape before removing the NAN values: (323, 9)
Shape after removing the NAN values: (203, 9)
After removing data which doesn't involve free G quad: (192, 9)
CPU times: user 10.5 s, sys: 28 ms, total: 10.5 s
Wall time: 10.6 s


In [15]:
y_pred

array([3., 3., 3., 3., 2., 1., 2., 2., 2., 1., 1., 1.])

In [29]:
conda install -c conda-forge jupyter_nbextensions_configurator


Solving environment: done

# All requested packages already installed.


Note: you may need to restart the kernel to use updated packages.
